In [0]:
# ==========================================
# E-Commerce Data Engineering Mini Project
# Bronze -> Silver -> Gold Architecture
# Technologies: PySpark | Spark SQL | Databricks
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, sum

# ------------------------------------------
# Create Spark Session
# ------------------------------------------
spark = SparkSession.builder \
    .appName("EcommerceDataEngineering") \
    .getOrCreate()

# ==========================================
# BRONZE LAYER (Raw Data Ingestion)
# ==========================================

bronze_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/default/databricks2027/Ecommerce_Sales_Data_2024_2025.csv")

print("========== BRONZE LAYER ==========")
print(f"Total Records : {bronze_df.count()}")

bronze_df.printSchema()
display(bronze_df)

# Save Bronze Layer
bronze_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/bronze"
)

# ==========================================
# SILVER LAYER (Data Cleaning & Transformation)
# ==========================================

silver_df = spark.read.parquet(
    "/Volumes/workspace/default/databricks2027/bronze"
)

# Remove Null Values
silver_df = silver_df.na.drop()

# Remove Duplicate Records
silver_df = silver_df.dropDuplicates()

# Create Calculated Columns
silver_df = silver_df.withColumn(
    "Total_Amount",
    col("Quantity") * col("Unit Price")
)

silver_df = silver_df.withColumn(
    "Discount_Percentage",
    col("Discount")
)

print("========== SILVER LAYER ==========")
print(f"Clean Records : {silver_df.count()}")

display(silver_df)

# Save Silver Layer
silver_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/silver"
)

# ==========================================
# GOLD LAYER (Business Ready Data)
# ==========================================

gold_df = spark.read.parquet(
    "/Volumes/workspace/default/databricks2027/silver"
)

gold_df.createOrReplaceTempView("sales")

print("========== GOLD LAYER ==========")

# Region-wise Sales
region_sales = spark.sql("""
SELECT
    Region,
    SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC
""")

print("Region Wise Sales")
display(region_sales)

# Category-wise Profit
category_profit = spark.sql("""
SELECT
    Category,
    SUM(Profit) AS Total_Profit
FROM sales
GROUP BY Category
ORDER BY Total_Profit DESC
""")

print("Category Wise Profit")
display(category_profit)

# Top 10 Products
top_products = spark.sql("""
SELECT
    `Product Name`,
    SUM(Sales) AS Revenue
FROM sales
GROUP BY `Product Name`
ORDER BY Revenue DESC
LIMIT 10
""")

print("Top Selling Products")
display(top_products)

# Monthly Sales
monthly_sales = spark.sql("""
SELECT
    month(`Order Date`) AS Month,
    SUM(Sales) AS Revenue
FROM sales
GROUP BY month(`Order Date`)
ORDER BY Month
""")

print("Monthly Revenue")
display(monthly_sales)

# Payment Mode Analysis
payment_mode = spark.sql("""
SELECT
    `Payment Mode`,
    COUNT(*) AS Total_Orders,
    SUM(Sales) AS Total_Sales
FROM sales
GROUP BY `Payment Mode`
ORDER BY Total_Sales DESC
""")

print("Payment Mode Analysis")
display(payment_mode)

# Save Gold Reports
region_sales.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/gold/region_sales"
)

category_profit.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/gold/category_profit"
)

top_products.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/gold/top_products"
)

monthly_sales.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/gold/monthly_sales"
)

payment_mode.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/databricks2027/gold/payment_mode"
)

print("========================================")
print("Mini Data Engineering Project Completed")
print("Bronze  -> Raw Data")
print("Silver  -> Cleaned & Transformed Data")
print("Gold    -> Business Reports")
print("========================================")